# Step 1 — Cellpose-SAM Segmentation, validated against `geff` ground truth

Goal: run Cellpose-SAM on a small local subset of the training data (one `44b6` sample,
one `6bba` sample) and check detection quality against the sparse point ground truth in
each sample's `.geff` graph.

**Important caveats (see `20260906_PLAN.txt` for full discussion):**
- Ground truth has **no instance masks** — only sparse point annotations (`nodes/props/{t,z,y,x}`).
  Validation here uses **centroid matching** (same scaled bipartite matching as the official
  eval metric: physical-µm distance, 7.0 µm cutoff), not IoU.
- Not every timepoint is labeled, so we only evaluate on timepoints that actually have GT nodes.
- A predicted cell with no matching GT point may be a real, simply-unannotated cell — treat
  precision/recall here as a *proxy* for segmentation quality, not the true competition metric.
- **No local GPU is available** (CPU-only torch in this env). This notebook runs on a small
  subset (a handful of labeled timepoints) to validate the pipeline logic on CPU in reasonable
  time. Full-scale segmentation across all samples/timepoints belongs on Kaggle's GPU notebook
  environment.


In [ ]:
import time

import numpy as np
import pandas as pd
import zarr
from scipy.ndimage import center_of_mass
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# physical scale used by the evaluation metric (confirmed in notebook/20260723_analysis.ipynb)
SCALE_Z, SCALE_Y, SCALE_X = 1.625, 0.40625, 0.40625
SCALE = np.array([SCALE_Z, SCALE_Y, SCALE_X])
MAX_MATCH_DIST_UM = 7.0

DATA_ROOT = "/mnt/d/KAGGLE/data/extracted/train"

# one sample from each dataset regime, already extracted to plain zarr/geff directories
SAMPLE_IDS = {
    "44b6": "44b6_0db75fae",
    "6bba": "6bba_2312ac41",
}

print("zarr", zarr.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)


## 1. Load image volume + geff ground truth for each sample

In [ ]:
def load_geff(sid):
    g = zarr.open_group(f"{DATA_ROOT}/{sid}.geff", mode="r")
    nodes = pd.DataFrame(
        {
            "node_id": g["nodes/ids"][:],
            "t": g["nodes/props/t/values"][:],
            "z": g["nodes/props/z/values"][:],
            "y": g["nodes/props/y/values"][:],
            "x": g["nodes/props/x/values"][:],
        }
    )
    edges = pd.DataFrame(g["edges/ids"][:], columns=["source", "target"])
    return nodes, edges


samples = {}
for prefix, sid in SAMPLE_IDS.items():
    img_group = zarr.open_group(f"{DATA_ROOT}/{sid}.zarr", mode="r")
    arr = img_group["0"]
    nodes, edges = load_geff(sid)
    samples[prefix] = {"sid": sid, "arr": arr, "nodes": nodes, "edges": edges}
    print(f"--- {prefix} ({sid}) ---")
    print("  volume shape (T,Z,Y,X):", arr.shape, "dtype:", arr.dtype)
    print("  GT nodes:", len(nodes), "edges:", len(edges), "labeled t range:", nodes['t'].min(), "-", nodes['t'].max())


## 2. Cellpose-SAM setup

Cellpose-SAM (cellpose >=4.0) is size-invariant, so no `diameter` estimation is required.

`do_3D=True` with `anisotropy=SCALE_Z/SCALE_X` was tried first, but is not CPU-feasible here:
Cellpose resamples Z to isotropic spacing (64 -> 256 planes at 4x anisotropy) and then forwards
the ViT/SAM backbone across those planes in multiple orthogonal views for the 3D flow
combination. A single-volume timing check showed this hadn't finished even its first
orthogonal-view pass after several minutes, and two full local runs (~6h each) never completed
the 6-volume loop. This machine has no GPU at all (confirmed via the Windows host, not just a
WSL/driver gap), so CPU is the only option.

Instead we use `do_3D=False` with `stitch_threshold`: segment each Z-plane independently in 2D
(one forward pass per plane, no resampling), then stitch same-cell masks across neighboring
slices into 3D instances by mask IoU. Cheaper by roughly the ratio of orthogonal-view passes
avoided, at the cost of true 3D-consistent flow fields — instance continuity across Z now
depends on the stitch IoU threshold instead.


In [ ]:
from cellpose import models

ANISOTROPY = SCALE_Z / SCALE_X
print("anisotropy (Z / XY):", ANISOTROPY)

model = models.CellposeModel(gpu=False)
print("model loaded:", type(model).__name__)


## 3. Run segmentation on a handful of labeled timepoints (CPU-feasible subset)

We restrict to timepoints that actually have GT annotations (so we can validate), and to a
small number of them per sample.

Even with `do_3D=False` + `stitch_threshold`, a full 256x256x64 volume measured ~2h on this
CPU-only, 4-core, memory-constrained (swap-exhausted) machine — not feasible for local
iteration. So each timepoint is additionally cropped to a small window centered on that
frame's GT points (capped size, margin comfortably larger than the 7 µm match cutoff) before
segmentation. This trades recall on any real cells outside the window for CPU feasibility;
predicted centroids are shifted back into full-volume voxel coordinates before GT matching so
the precision/recall numbers stay comparable.
</cell id="cd122ece">


In [ ]:
from pathlib import Path

N_TIMEPOINTS_PER_SAMPLE = 3
STITCH_THRESHOLD = 0.25  # cross-slice mask IoU required to link 2D detections into one 3D instance

# crop window caps (voxels) -- keeps per-volume compute bounded on this CPU-only machine.
# margins are comfortably larger than the 7um / [SCALE_Z, SCALE_Y, SCALE_X] match cutoff.
MAX_Z, MAX_Y, MAX_X = 24, 144, 144
MARGIN_Z, MARGIN_YX = 6, 24

CHECKPOINT_DIR = Path("_checkpoints/01_cellpose_sam")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def crop_window(center, margin, cap, extent):
    """1D window of size min(2*margin, cap) centered on `center`, clipped to [0, extent)."""
    half = min(margin, cap // 2)
    lo = int(np.clip(center - half, 0, max(0, extent - 2 * half)))
    hi = min(extent, lo + 2 * half)
    return lo, hi


masks_by_sample = {}
crop_offsets = {}
for prefix, s in samples.items():
    labeled_ts = sorted(s["nodes"]["t"].unique())
    chosen_ts = labeled_ts[:: max(1, len(labeled_ts) // N_TIMEPOINTS_PER_SAMPLE)][:N_TIMEPOINTS_PER_SAMPLE]
    print(f"{prefix}: {len(labeled_ts)} labeled timepoints available, using {chosen_ts}")

    Z, Y, X = s["arr"].shape[1:]
    frame_masks = {}
    frame_offsets = {}
    for t in chosen_ts:
        gt_rows = s["nodes"][s["nodes"]["t"] == t]
        z0, z1 = crop_window((gt_rows["z"].min() + gt_rows["z"].max()) / 2, MARGIN_Z, MAX_Z, Z)
        y0, y1 = crop_window((gt_rows["y"].min() + gt_rows["y"].max()) / 2, MARGIN_YX, MAX_Y, Y)
        x0, x1 = crop_window((gt_rows["x"].min() + gt_rows["x"].max()) / 2, MARGIN_YX, MAX_X, X)
        offset = (z0, y0, x0)

        ckpt_path = CHECKPOINT_DIR / f"{prefix}_t{int(t)}.npz"
        if ckpt_path.exists():
            data = np.load(ckpt_path)
            masks = data["masks"]
            offset = tuple(data["offset"])
            print(f"  t={t}: loaded cached masks -> {masks.max()} instances")
        else:
            vol = np.asarray(s["arr"][int(t), z0:z1, y0:y1, x0:x1])  # (Z, Y, X) crop
            t0 = time.time()
            masks, flows, styles = model.eval(
                vol, do_3D=False, stitch_threshold=STITCH_THRESHOLD, z_axis=0, diameter=None
            )
            dt = time.time() - t0
            np.savez(ckpt_path, masks=masks, offset=np.array(offset))
            print(f"  t={t}: crop {vol.shape} offset={offset} -> {masks.max()} instances in {dt:.1f}s")
        frame_masks[int(t)] = masks
        frame_offsets[int(t)] = offset
    masks_by_sample[prefix] = frame_masks
    crop_offsets[prefix] = frame_offsets


## 4. Extract predicted centroids and match against GT (scaled bipartite matching, 7 µm)

In [ ]:
def predicted_centroids_um(masks, offset=(0, 0, 0)):
    labels = np.unique(masks)
    labels = labels[labels != 0]
    if len(labels) == 0:
        return np.empty((0, 3))
    centers = center_of_mass(masks, labels=masks, index=labels)  # list of (z,y,x) in voxels, crop-local
    centers = np.asarray(centers) + np.asarray(offset)  # shift back to full-volume voxel coords
    return centers * SCALE


def gt_centroids_um(nodes_df, t):
    rows = nodes_df[nodes_df["t"] == t]
    return rows[["z", "y", "x"]].to_numpy(dtype=float) * SCALE


def match_centroids(pred_um, gt_um, max_dist=MAX_MATCH_DIST_UM):
    if len(pred_um) == 0 or len(gt_um) == 0:
        return [], list(range(len(pred_um))), list(range(len(gt_um)))
    cost = cdist(pred_um, gt_um)
    row_ind, col_ind = linear_sum_assignment(cost)
    matches = [(r, c) for r, c in zip(row_ind, col_ind) if cost[r, c] <= max_dist]
    matched_pred = {r for r, _ in matches}
    matched_gt = {c for _, c in matches}
    fp = [r for r in range(len(pred_um)) if r not in matched_pred]
    fn = [c for c in range(len(gt_um)) if c not in matched_gt]
    return matches, fp, fn


results = []
for prefix, frame_masks in masks_by_sample.items():
    nodes_df = samples[prefix]["nodes"]
    offsets = crop_offsets[prefix]
    for t, masks in frame_masks.items():
        pred_um = predicted_centroids_um(masks, offsets[t])
        gt_um = gt_centroids_um(nodes_df, t)
        matches, fp, fn = match_centroids(pred_um, gt_um)
        results.append(
            {
                "prefix": prefix,
                "t": t,
                "n_pred": len(pred_um),
                "n_gt": len(gt_um),
                "TP": len(matches),
                "FP": len(fp),
                "FN": len(fn),
            }
        )

results_df = pd.DataFrame(results)
results_df


## 5. Aggregate precision / recall / F1 per dataset regime

In [ ]:
def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    return precision, recall, f1


summary_rows = []
for prefix, sub in results_df.groupby("prefix"):
    tp, fp, fn = sub["TP"].sum(), sub["FP"].sum(), sub["FN"].sum()
    precision, recall, f1 = precision_recall_f1(tp, fp, fn)
    summary_rows.append(
        {"prefix": prefix, "TP": tp, "FP": fp, "FN": fn, "precision": precision, "recall": recall, "f1": f1}
    )

summary_df = pd.DataFrame(summary_rows)
summary_df


## 6. Next steps

- Review the precision/recall/F1 numbers above per dataset regime (`44b6` vs `6bba`).
- If segmentation quality looks reasonable, proceed to Step 2 (per-cell feature tokens) using
  these predicted masks as the source of instances.
- If not, consider tuning Cellpose-SAM (e.g. `flow_threshold`, `cellprob_threshold`) or
  reconsidering the anisotropy handling (resampling Z to more isotropic spacing before
  segmentation) before moving on.
- Full-scale runs (all samples, all timepoints) should happen on Kaggle's GPU notebook
  environment, not this local CPU env — see `20260906_PLAN.txt` for the full design discussion.
